# Experiment Runner

Notebook interface for `experiments/runner.py`.

This notebook avoids Drive-backed virtualenvs. It uses the active notebook Python interpreter, which works in Colab, local Jupyter, and other hosted notebook runtimes.

**Disconnect-safe**: the runner writes and verifies each child JSON payload under the configured run directory as soon as that experiment finishes. `summary.json` is refreshed during the run, so a stopped Colab session still leaves the latest completed records on Drive.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import os
import sys

IN_COLAB = importlib.util.find_spec('google.colab') is not None

# --- Storage / repo controls ---
USE_GOOGLE_DRIVE = IN_COLAB
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments' if IN_COLAB else str(Path.cwd() / 'notebook-runs')
REPO_URL = 'https://github.com/hotz99/loss-grid-computer.git'
REPO_DIR = '/content/loss-grid-computer' if IN_COLAB else str(Path.cwd())
ASSETS_DIR = f'{DRIVE_ROOT}/assets'

# Leave False unless the runtime is missing dependencies. In Colab, torch is usually preinstalled.
INSTALL_REQUIREMENTS = False
LINK_DRIVE_ASSETS = True

RUN_LABEL = 'experiment_runner'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print({'in_colab': IN_COLAB, 'repo_dir': REPO_DIR, 'drive_root': DRIVE_ROOT, 'run_id': RUN_ID, 'python': sys.executable})


## Prepare Runtime

Mount Drive when available, clone or update the repo, optionally install requirements into the current kernel, and expose Drive assets to the repo.


In [ ]:
import shutil
import subprocess

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(ASSETS_DIR).mkdir(parents=True, exist_ok=True)

repo_path = Path(REPO_DIR)
if (repo_path / '.git').exists():
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
elif not (repo_path / 'experiments' / 'runner.py').exists():
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', REPO_URL, REPO_DIR])
else:
    print(f'Using existing repo checkout: {REPO_DIR}')

if INSTALL_REQUIREMENTS:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_path / 'requirements.txt')])

repo_assets = repo_path / 'assets'
drive_assets = Path(ASSETS_DIR)
if LINK_DRIVE_ASSETS and drive_assets.exists():
    if repo_assets.is_symlink():
        repo_assets.unlink()
    if not repo_assets.exists():
        repo_assets.symlink_to(drive_assets, target_is_directory=True)
        print(f'Linked assets: {repo_assets} -> {drive_assets}')
    else:
        print(f'Keeping existing repo assets directory: {repo_assets}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'mps': hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()})


## Configure Runner

Edit these globals, then execute the run cell. Registry booleans are the experiment toggles.


In [ ]:
import experiments.runner as aio

RUN_DIR = Path(DRIVE_ROOT) / 'runs' / f'{RUN_LABEL}-{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

aio.DEVICE = 'auto'
aio.OUTPUT_DIR = str(RUN_DIR)
aio.FAIL_FAST = False
aio.VERBOSE_EXPERIMENT_LOGS = False

aio.SEED = 1337
aio.SAMPLE_COUNT = 1024

# Experiment A/B bounded benchmark grid. Experiment C uses its own full-session grid below.
aio.GRID_RESOLUTION = 8
aio.EXPERIMENT_C_SESSION_GRID_RESOLUTION = 40
aio.GRID_SCALE = 1.0
aio.GPU_BATCH_SIZE = 64
aio.ATOL = 1e-6
aio.RTOL = 1e-5

# Experiment C (RQ3) operating point. Default `None` makes Exp C read Exp B's
# parity_probe for the selected workload and apply the plan's selection rule:
# native `s = 1` if Exp B records `hybrid_wins` at slowdown 1.0, else Exp B's
# `slowdown_used`. Setting a float overrides the rule (research escape hatch).
# The chosen `s` is applied uniformly to calibration, cached-policy runs, and
# vanilla runs within the session. `OPERATING_POINT_SOURCE = None` lets Exp C
# auto-fill the provenance string from the selection branch taken.
aio.EXPERIMENT_C_WORKLOAD = 'cifar10_row_gru_classification'
aio.EXPERIMENT_C_GPU_SLOWDOWN = None
aio.EXPERIMENT_C_OPERATING_POINT_SOURCE = None

aio.CALIBRATION_RETRY = 3
aio.MAX_CPU_WORKER_CANDIDATE = None

aio.RUN_LABEL = RUN_LABEL
aio.MLTASK_WORKLOADS = list(aio.DEFAULT_FUNCTIONAL_EVAL_WORKLOADS)
aio.FUNCTIONAL_EVAL_WORKLOADS = None
aio.FUNCTIONAL_EVAL_SAMPLE_COUNTS = [1024]
aio.FUNCTIONAL_EVAL_REPEATS = 3
aio.FUNCTIONAL_EVAL_BATCH_SIZE = 32
aio.POINT_CHUNK_SIZES = [32, 64]
aio.MAX_MEMORY_FRACTION = 0.85
aio.INCLUDE_VMAP_REPRODUCTION = False
aio.INCLUDE_FULL_TEST_SET = False

aio.EXPERIMENT_REGISTRY['e0_platform_inventory']['enabled'] = True
aio.EXPERIMENT_REGISTRY['functional_eval_api_probe']['enabled'] = True
aio.EXPERIMENT_REGISTRY['experiment_a_profiling']['enabled'] = True
aio.EXPERIMENT_REGISTRY['experiment_a_candidates']['enabled'] = True
aio.EXPERIMENT_REGISTRY['experiment_b_hybrid_applicability']['enabled'] = True
aio.EXPERIMENT_REGISTRY['experiment_c_calibration_cache']['enabled'] = True
aio.EXPERIMENT_REGISTRY['statistical_analysis']['enabled'] = True
aio.EXPERIMENT_REGISTRY['experiment_d_merged_stack']['enabled'] = False
aio.EXPERIMENT_REGISTRY['progressive_visualization_deferred']['enabled'] = False

print('Output directory:', aio.OUTPUT_DIR)
print('Grid policy:', {'exp_a_b_grid': aio.GRID_RESOLUTION, 'exp_c_session_grid': aio.EXPERIMENT_C_SESSION_GRID_RESOLUTION})
print('Experiment C:', {'workload': aio.EXPERIMENT_C_WORKLOAD, 'gpu_slowdown': aio.EXPERIMENT_C_GPU_SLOWDOWN, 'operating_point_source': aio.EXPERIMENT_C_OPERATING_POINT_SOURCE})
print('Enabled experiments:', [name for name, entry in aio.EXPERIMENT_REGISTRY.items() if entry['enabled']])


## Run Suite

The runner writes verified JSON directly under `aio.OUTPUT_DIR`: one child payload per registry entry, a refreshed `summary.json`, and partial payloads for long composed entries. `summary['records']` is a direct composition of each child payload's `record`, with runner-added `duration_s` and `output_path`.


In [ ]:
import json
from pathlib import Path

summary = aio.run_aio_suite()
summary_path = Path(summary['output_path'])
loaded_summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert loaded_summary['status'] == 'completed', loaded_summary.get('status')
assert loaded_summary['records'] == summary['records']

child_paths = {
    name: Path(record['output_path'])
    for name, record in summary['records'].items()
    if record.get('output_path')
}
missing = [str(path) for path in child_paths.values() if not path.exists()]
assert not missing, missing
for path in child_paths.values():
    json.loads(path.read_text(encoding='utf-8'))

print('summary:', summary_path)
print('verified child payloads:', len(child_paths))
print({name: payload.get('status') for name, payload in summary['experiments'].items()})
print({name: str(path) for name, path in child_paths.items()})


## Inspect Minimal Records

These records are the paper-facing log. They should match the `record` object persisted in each child payload after the runner adds `duration_s` and `output_path`.


In [ ]:
import json
from pathlib import Path

records = summary.get('records', {})
print(json.dumps(records, indent=2, sort_keys=True))

mismatches = {}
for name, record in records.items():
    output_path = record.get('output_path')
    if not output_path:
        continue
    payload = json.loads(Path(output_path).read_text())
    if payload.get('record') != record:
        mismatches[name] = {
            'summary_record': record,
            'child_record': payload.get('record'),
        }

print('record mismatches:', list(mismatches))
assert not mismatches, json.dumps(mismatches, indent=2, sort_keys=True)
